In [1]:
import timeit
from timeit import Timer
from timeit import default_timer as timer

import numpy as np
from numba import njit
from numba.openmp import openmp_context as openmp
from numba.openmp import omp_get_wtime
from numba.openmp import omp_get_thread_num, omp_get_num_threads, omp_set_num_threads


/blue/m2qm-efrc/shlufl/condaevs/python3.9.5/lib/python3.9/site-packages/numba/__init__.py:172: UserWarning: llvmlite version format not recognized!
  warnings.warn("llvmlite version format not recognized!")


In [26]:
@njit
def dgemm(nthread, order):
    omp_set_num_threads(nthread)
    # allocate and initialize arrays
    A = np.zeros((order,order))
    B = np.zeros((order,order))
    C = np.zeros((order,order))
    # Assign values to A and B such that
    # the product matrix has a known value.
    for i in range(order):
        A[:,i] = float(i)
        B[:,i] = float(i)
    tInit = omp_get_wtime()
    with openmp("parallel for private(j,k)"):
        for i in range(order):
            for k in range(order):
                for j in range(order):
                    C[i][j] += A[i][k] * B[k][j]
    dgemmTime = omp_get_wtime() - tInit
    # Check result
    checksum = 0.0;
    for i in range(order):
        for j in range(order):
            checksum += C[i][j];
    ref_checksum = order*order*order
    ref_checksum *= 0.25*(order-1.0)*(order-1.0)
    eps=1.e-8
    if abs((checksum - ref_checksum)/ref_checksum) < eps:
        print('Solution validates')
        nflops = 2.0*order*order*order
        print('Rate (MF/s): ',1.e-6*nflops/dgemmTime)


In [32]:
dgemm(1, 1000) # Rate (MF/s): 1733.0781973161133
# dgemm(2, 1000) # Rate (MF/s): 3471.667690957888
# dgemm(4, 1000) # Rate (MF/s): 6860.707792489264
# dgemm(8, 1000) # Rate (MF/s): 13627.604132822145


Solution validates
Rate (MF/s):  1737.0491231623103


In [2]:
@njit
def matrix_multiply(nthread, n, A, B, C):
    omp_set_num_threads(nthread)

    with openmp("parallel for private(j,k)"):
        for i in range(n):
            for k in range(n):
                for j in range(n):
                    C[i][j] += A[i][k] * B[k][j]

    return


In [14]:
n = 32 

x = np.ones((n,n), dtype=np.complex128)
y = np.ones((n,n), dtype=np.complex128)
z = np.zeros((n,n), dtype=np.complex128)

# number=10**6, 52.23124236613512 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
# t = Timer(lambda: matrix_multiply(1, n, x, y, z))
# print(t.timeit(number=10**6))

# number=10**6, 15.498567640315741 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
# t = Timer(lambda: matrix_multiply(4, n, x, y, z))
# print(t.timeit(number=10**6))

# number=10**6, 11.55068068485707 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
# t = Timer(lambda: matrix_multiply(8, n, x, y, z))
# print(t.timeit(number=10**6))

# number=10**6, 10.51526900700992 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
# t = Timer(lambda: matrix_multiply(16, n, x, y, z))
# print(t.timeit(number=10**6))

# number=10**6, 7.881256141001359 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
t = Timer(lambda: matrix_multiply(32, n, x, y, z))
print(t.timeit(number=10**6))


7.165125391969923


In [15]:
# number=10**6, 25.718136625830084 seconds, on hipergator 2.95 GHz AMD EPYC 75F3 32-Core Processor
t = Timer(lambda: np.matmul(x, y))
print(t.timeit(number=10**6))


29.274721321999095
